In [5]:
import gc
import json
from pathlib import Path

import pandas as pd
import torch
from datasets import Dataset
from peft import LoraConfig, TaskType, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer

In [6]:
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
OUTPUT_DIR = Path("agents/models/trained/model_qwen_solver_fast")
MAX_SEQ_LEN = 512

SYSTEM_PROMPT = (
    "You are a Kubernetes Site Reliability Engineering (SRE) agent. "
    "Given raw observability evidence from a Kubernetes incident, provide:\n"
    "1. A root cause diagnosis explaining what went wrong and why.\n"
    "2. A step-by-step fix plan to resolve the incident.\n"
    "3. Concrete actions or commands to apply the fix.\n"
    "4. Verification steps to confirm the fix worked.\n"
    "5. Rollback guidance if the fix causes issues."
)

In [7]:
def build_evidence_prompt(row):
    return str(row.get("evidence_text", "")).strip()


def build_target_response(row):
    sections = []

    diagnosis = str(row.get("diagnosis_text", "")).strip()
    fix_plan = str(row.get("fix_plan_text", "")).strip()
    actions = str(row.get("actions_text", "")).strip()
    verification = str(row.get("verification_text", "")).strip()
    rollback = str(row.get("rollback_text", "")).strip()

    if diagnosis:
        sections.append(f"## Diagnosis\n{diagnosis}")
    if fix_plan:
        sections.append(f"## Fix Plan\n{fix_plan}")
    if actions:
        sections.append(f"## Actions\n{actions}")
    if verification:
        sections.append(f"## Verification\n{verification}")
    if rollback:
        sections.append(f"## Rollback\n{rollback}")

    return "\n\n".join(sections)

def build_chat_example(row):
    evidence = build_evidence_prompt(row)
    target = build_target_response(row)
    return (
        f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
        f"<|im_start|>user\n"
        f"Analyze this Kubernetes incident and provide diagnosis, remediation, verification, and rollback guidance.\n\n"
        f"Incident Evidence:\n{evidence}<|im_end|>\n"
        f"<|im_start|>assistant\n{target}<|im_end|>"
    )

In [8]:
df = pd.read_parquet("master_incident_dataset.parquet")
df = df[df["scenario_id"] != "dns_resolution_failure"].copy()
df = df.reset_index(drop=True)

target_cols = [
    "diagnosis_text",
    "fix_plan_text",
    "actions_text",
    "verification_text",
    "rollback_text",
]

df["evidence_text"] = df["evidence_text"].fillna("").astype(str)
for col in target_cols:
    df[col] = df[col].fillna("").astype(str)

df = df[df["evidence_text"].str.strip() != ""].copy()
df = df[df[target_cols].apply(lambda x: x.str.strip()).ne("").any(axis=1)].copy()
df = df.reset_index(drop=True)

texts = [build_chat_example(row) for row in df.to_dict("records")]
dataset = Dataset.from_pandas(pd.DataFrame({"text": texts}), preserve_index=False)
split_ds = dataset.train_test_split(test_size=0.2, seed=42)

print("Train:", len(split_ds["train"]))
print("Test:", len(split_ds["test"]))

Train: 6800
Test: 1700


In [9]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id


# ---------------------------
# Pre-tokenize once
# ---------------------------
def tokenize_fn(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_SEQ_LEN,
        padding=False,
    )

tokenized_train = split_ds["train"].map(
    tokenize_fn,
    batched=True,
    remove_columns=["text"],
    desc="Tokenizing train",
)

tokenized_test = split_ds["test"].map(
    tokenize_fn,
    batched=True,
    remove_columns=["text"],
    desc="Tokenizing test",
)

tokenized_train.set_format("torch")
tokenized_test.set_format("torch")

Tokenizing train:   0%|          | 0/6800 [00:00<?, ? examples/s]

Tokenizing test:   0%|          | 0/1700 [00:00<?, ? examples/s]

In [10]:
import numpy as np

enc = tokenizer(texts, add_special_tokens=True, truncation=False)
lengths = [len(x) for x in enc["input_ids"]]

print("min:", min(lengths))
print("mean:", sum(lengths) / len(lengths))
print("p50:", np.percentile(lengths, 50))
print("p90:", np.percentile(lengths, 90))
print("p95:", np.percentile(lengths, 95))
print("max:", max(lengths))

min: 609
mean: 753.7541176470588
p50: 756.0
p90: 858.0
p95: 884.0
max: 963


In [11]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.bfloat16,
    trust_remote_code=True,
)

model.config.use_cache = False
model.gradient_checkpointing_enable()

lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

model = get_peft_model(model, lora_config)
trainable, total = model.get_nb_trainable_parameters()
print(f"Trainable params: {trainable:,} / {total:,} ({100 * trainable / total:.2f}%)")

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Trainable params: 20,185,088 / 7,635,801,600 (0.26%)


In [12]:
training_args = SFTConfig(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=1e-4,
    weight_decay=0.01,
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    logging_steps=50,
    eval_strategy="no",
    save_strategy="no",
    bf16=True,
    fp16=False,
    report_to="none",
    max_length=1024,
    gradient_checkpointing=False,
    dataloader_pin_memory=True,
    dataloader_num_workers=2,
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=tokenized_train,
    eval_dataset=None,
    args=training_args,
)

print("\nStarting training...")
trainer.train()

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
adapter_path = OUTPUT_DIR / "lora_adapter"
model.save_pretrained(str(adapter_path))
tokenizer.save_pretrained(str(adapter_path))


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Truncating train dataset:   0%|          | 0/6800 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.



Starting training...


Step,Training Loss
50,1.449841
100,0.405964
150,0.189993
200,0.174597
250,0.169718
300,0.166496
350,0.164787
400,0.163685
450,0.163817
500,0.162693


('agents/models/trained/model_qwen_solver_fast/lora_adapter/tokenizer_config.json',
 'agents/models/trained/model_qwen_solver_fast/lora_adapter/chat_template.jinja',
 'agents/models/trained/model_qwen_solver_fast/lora_adapter/tokenizer.json')

In [13]:
with open(OUTPUT_DIR / "training_metadata.json", "w") as f:
    json.dump(
        {
            "model_id": MODEL_ID,
            "max_seq_len": MAX_SEQ_LEN,
            "lora_r": 32,
            "lora_alpha": 64,
            "train_rows": len(tokenized_train),
            "test_rows": len(tokenized_test),
        },
        f,
        indent=2,
    )

print("Done.")

Done.


In [1]:

import gc
import torch

gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

print(torch.cuda.memory_allocated() / 1024**3, "GB allocated")
print(torch.cuda.memory_reserved() / 1024**3, "GB reserved")

0.0 GB allocated
0.0 GB reserved
